# Statistical Tests – Vanguard A/B Test
All tests run from the exported CSVs, so this notebook is fully independent
of the data-loading pipeline in `Error_rate_Claire.ipynb`.

**Inputs** (from `../data/clean/`)
- `df_alldata.csv` – enriched client table with `group`, `completed`,
  `made_error`, `n_errors`, `completion_seconds`, `clnt_age`, `bal`
- `funnel_by_step.csv` – step-level reach counts & rates
- `errors_by_transition.csv` – transition-level error counts & rates

**Tests performed**
1. Overall completion-rate z-test (test vs control)
2. Per-step reach-rate z-tests (funnel)
3. Per-transition error-rate z-tests
4. Within-group chi-square (funnel homogeneity)
5. Completion rate by age quintile (z-test)
6. Error rate by age quintile (z-test)
7. Mean error count by age quintile (two-sample z-test)
8. Completion rate by balance quintile (z-test)
9. Numeric correlation matrix
10. Gender effects


## 1  Imports

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency

## 2  Load CSVs

In [2]:
DATA = '../data/clean/'

df_cd      = pd.read_csv(DATA + 'df_alldata.csv')
df_funnel  = pd.read_csv(DATA + 'funnel_by_step.csv')
df_errors  = pd.read_csv(DATA + 'errors_by_transition.csv')

# Convenience views
ctrl = df_cd[df_cd['group'] == 'control']
test = df_cd[df_cd['group'] == 'test']

print(f"Control n={len(ctrl):,}  |  Test n={len(test):,}")

Control n=22,934  |  Test n=25,388


## 3  Overall completion-rate z-test
Did the new UI change the share of clients who finished the process?


In [3]:
# Pull start and confirm counts from the funnel CSV
def funnel_count(group, step):
    row = df_funnel[(df_funnel['group'] == group) & (df_funnel['step'] == step)]
    return int(row['clients_reached'].iloc[0])

n_start_ctrl  = funnel_count('control', 'start')
n_start_test  = funnel_count('test',    'start')
n_conf_ctrl   = funnel_count('control', 'confirm')
n_conf_test   = funnel_count('test',    'confirm')

stat, pval = proportions_ztest(
    [n_conf_test, n_conf_ctrl],
    [n_start_test, n_start_ctrl]
)

print(f"Control completion rate : {n_conf_ctrl / n_start_ctrl:.4f}")
print(f"Test    completion rate : {n_conf_test / n_start_test:.4f}")
print(f"z = {stat:.3f}  |  p = {pval:.4f}")

Control completion rate : 0.5556
Test    completion rate : 0.6043
z = 10.836  |  p = 0.0000


In [19]:
n_start_ctrl, n_start_test

(22934, 25388)

## 4  Per-step reach-rate z-tests
Test whether each funnel step shows a significant difference in reach rate
between groups.  Denominator is each group's `start` count.


In [15]:
STEPS = ['step_1', 'step_2', 'step_3', 'confirm']

rows = []
for step in STEPS:
    c = funnel_count('control', step)
    t = funnel_count('test',    step)
    stat, pval = proportions_ztest([t, c], [n_start_test, n_start_ctrl])
    rows.append({'step': step,
                 'ctrl_rate': round(c / n_start_ctrl, 4),
                 'test_rate': round(t / n_start_test, 4),
                 'z_stat':    round(stat, 3),
                 'p_value':   pval})
                 # 'p_value':   round(pval, 4)})

pd.DataFrame(rows)

,step,ctrl_rate,test_rate,z_stat,p_value
0,step_1,0.7909,0.8664,22.097,3.371475e-108
1,step_2,0.7034,0.7682,16.162,9.309575e-59
2,step_3,0.6449,0.7028,13.568,6.204697e-42
3,confirm,0.5556,0.6043,10.836,2.321903e-27


## 5  Per-transition error-rate z-tests
Error rate = clients who hit the regression pattern / all clients who started.


In [5]:
error_rows = []
for _, row in df_errors.groupby('transition'):
    c_row = df_errors[(df_errors['transition'] == _) & (df_errors['group'] == 'control')].iloc[0]
    t_row = df_errors[(df_errors['transition'] == _) & (df_errors['group'] == 'test')].iloc[0]

    stat, pval = proportions_ztest(
        [t_row['error_count'], c_row['error_count']],
        [n_start_test, n_start_ctrl]
    )
    error_rows.append({
        'transition':  _,
        'ctrl_rate':   round(c_row['error_rate'], 4),
        'test_rate':   round(t_row['error_rate'], 4),
        'z_stat':      round(stat, 3),
        'p_value':     round(pval, 4)
    })

pd.DataFrame(error_rows)

,transition,ctrl_rate,test_rate,z_stat,p_value
0,start→step_1,0.0664,0.1488,28.932,0.0000
1,step_1→step_2,0.0360,0.0498,8.617,0.0000
2,step_2→step_3,0.0755,0.0517,-7.400,0.0000
3,step_3→confirm,0.0000,0.0000,0.950,0.3419


## 6  Within-group chi-square (funnel homogeneity)
Tests whether drop-off is uniform across funnel steps within each group.
A significant result means some transitions lose disproportionately more users.


In [6]:
def within_group_chi2(group_label):
    counts = [funnel_count(group_label, s)
              for s in ['start', 'step_1', 'step_2', 'step_3', 'confirm']]
    dropped   = [counts[i-1] - counts[i] for i in range(1, len(counts))]
    continued = counts[1:]
    stat, pval, dof, _ = chi2_contingency([continued, dropped])
    print(f"{group_label.capitalize():>8}:  chi2={stat:.3f}  p={pval:.4f}  dof={dof}")

within_group_chi2('control')
within_group_chi2('test')

 Control:  chi2=1456.592  p=0.0000  dof=3
    Test:  chi2=347.433  p=0.0000  dof=3


## 7  Age-stratified tests
Clients are split into quintiles by age.  Three outcomes are tested within
each quintile: completion rate, error rate, and mean error count.


In [7]:
# Create age quintile column (drop duplicate bin edges if they exist)
df_cd2 = df_cd.copy()
df_cd2['age_bin_q'] = pd.qcut(df_cd2['clnt_age'], q=5, duplicates='drop')

In [8]:
def ztest_proportion_by_bin(df, bin_col, outcome):
    """
    Two-proportion z-test (test vs control) for a binary outcome,
    run independently within each bin of `bin_col`.
    """
    rows = []
    for bin_val, sub in df.groupby(bin_col):
        t = sub[sub['group'] == 'test']
        c = sub[sub['group'] == 'control']
        if len(t) == 0 or len(c) == 0:
            continue
        stat, pval = proportions_ztest(
            [t[outcome].sum(), c[outcome].sum()],
            [len(t), len(c)]
        )
        rows.append({bin_col: bin_val,
                     'test_rate':    round(t[outcome].mean(), 4),
                     'control_rate': round(c[outcome].mean(), 4),
                     'abs_diff':     round(t[outcome].mean() - c[outcome].mean(), 4),
                     'z_stat':       round(stat, 3),
                     'p_value':      round(pval, 4)})
    return pd.DataFrame(rows)


def ztest_mean_by_bin(df, bin_col, outcome):
    """
    Two-sample mean z-test for a continuous/count outcome,
    run independently within each bin of `bin_col`.
    """
    rows = []
    for bin_val, sub in df.groupby(bin_col):
        t = sub[sub['group'] == 'test'][outcome].dropna()
        c = sub[sub['group'] == 'control'][outcome].dropna()
        if len(t) < 2 or len(c) < 2:
            continue
        se = np.sqrt(t.var(ddof=1)/len(t) + c.var(ddof=1)/len(c))
        z  = (t.mean() - c.mean()) / se if se > 0 else np.nan
        p  = 2 * (1 - stats.norm.cdf(abs(z))) if se > 0 else np.nan
        rows.append({bin_col:          bin_val,
                     'test_mean':      round(t.mean(), 4),
                     'control_mean':   round(c.mean(), 4),
                     'z_stat':         round(z, 3),
                     'p_value':        round(p, 4)})
    return pd.DataFrame(rows)

In [9]:
print("=== Completion rate by age quintile ===")
display(ztest_proportion_by_bin(df_cd2, 'age_bin_q', 'completed'))

print("\n=== Error rate by age quintile ===")
display(ztest_proportion_by_bin(df_cd2, 'age_bin_q', 'made_error'))

print("\n=== Mean error count by age quintile ===")
display(ztest_mean_by_bin(df_cd2, 'age_bin_q', 'n_errors'))

=== Completion rate by age quintile ===


,age_bin_q,test_rate,control_rate,abs_diff,z_stat,p_value
0,"(16.999, 31.0]",0.6555,0.5795,0.0760,7.753,0.0000
1,"(31.0, 42.0]",0.6319,0.5957,0.0362,3.634,0.0003
2,"(42.0, 53.0]",0.6104,0.5725,0.0379,3.859,0.0001
3,"(53.0, 62.0]",0.5784,0.5457,0.0327,3.227,0.0012
4,"(62.0, 96.0]",0.5393,0.4833,0.0560,5.390,0.0000



=== Error rate by age quintile ===


,age_bin_q,test_rate,control_rate,abs_diff,z_stat,p_value
0,"(16.999, 31.0]",0.1915,0.1426,0.0489,6.470,0.0
1,"(31.0, 42.0]",0.1712,0.1185,0.0527,7.261,0.0
2,"(42.0, 53.0]",0.1984,0.1338,0.0647,8.665,0.0
3,"(53.0, 62.0]",0.2254,0.1357,0.0897,11.360,0.0
4,"(62.0, 96.0]",0.2331,0.1447,0.0884,10.839,0.0



=== Mean error count by age quintile ===


,age_bin_q,test_mean,control_mean,z_stat,p_value
0,"(16.999, 31.0]",0.2005,0.1493,6.368,0.0
1,"(31.0, 42.0]",0.1817,0.1240,7.446,0.0
2,"(42.0, 53.0]",0.2113,0.1394,8.960,0.0
3,"(53.0, 62.0]",0.2452,0.1418,11.988,0.0
4,"(62.0, 96.0]",0.2494,0.1487,11.543,0.0


## 8  Balance-stratified completion-rate test
Same approach as above but split by account balance quintile.


In [10]:
df_cd2['bal_bin_q'] = pd.qcut(df_cd2['bal'], q=5, duplicates='drop')

print("=== Completion rate by balance quintile ===")
display(ztest_proportion_by_bin(df_cd2, 'bal_bin_q', 'completed'))

=== Completion rate by balance quintile ===


,bal_bin_q,test_rate,control_rate,abs_diff,z_stat,p_value
0,"(23789.439, 36002.1]",0.6254,0.5756,0.0498,4.986,0.0
1,"(36002.1, 52863.75]",0.6125,0.5699,0.0426,4.254,0.0
2,"(52863.75, 83533.54]",0.6062,0.5585,0.0477,4.750,0.0
3,"(83533.54, 170235.84]",0.5893,0.5369,0.0523,5.180,0.0
4,"(170235.84, 8986419.77]",0.5876,0.5376,0.0500,4.950,0.0


## 9  Numeric correlation matrix
Quick sanity-check on relationships between numeric features and outcomes.


In [18]:
df_cd.corr(numeric_only=True).round(3)

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,num_accts,bal,calls_6_mnth,logons_6_mnth,made_error,n_errors,completed,completion_minutes,completion_seconds
client_id,1.000,0.007,0.007,-0.001,-0.002,-0.001,-0.006,-0.006,-0.002,-0.002,-0.001,-0.002,-0.002
clnt_tenure_yr,0.007,1.000,0.999,0.307,0.143,0.203,0.045,0.049,-0.003,-0.002,-0.029,0.034,0.034
clnt_tenure_mnth,0.007,0.999,1.000,0.307,0.143,0.202,0.043,0.047,-0.002,-0.002,-0.030,0.035,0.035
clnt_age,-0.001,0.307,0.307,1.000,-0.030,0.211,0.045,0.053,0.030,0.031,-0.092,0.186,0.186
num_accts,-0.002,0.143,0.143,-0.030,1.000,0.263,0.233,0.238,-0.025,-0.027,0.029,-0.050,-0.050
bal,-0.001,0.203,0.202,0.211,0.263,1.000,0.169,0.176,-0.005,-0.005,-0.014,0.028,0.028
calls_6_mnth,-0.006,0.045,0.043,0.045,0.233,0.169,1.000,0.995,-0.018,-0.017,-0.019,-0.038,-0.038
logons_6_mnth,-0.006,0.049,0.047,0.053,0.238,0.176,0.995,1.000,-0.016,-0.016,-0.021,-0.035,-0.035
made_error,-0.002,-0.003,-0.002,0.030,-0.025,-0.005,-0.018,-0.016,1.000,0.971,-0.012,0.054,0.054
n_errors,-0.002,-0.002,-0.002,0.031,-0.027,-0.005,-0.017,-0.016,0.971,1.000,-0.012,0.058,0.058


## 10  Gender Effects
Is there a difference between genders, do both improve with the Test UI?


In [22]:
df_gender = df_cd[df_cd['gendr'].isin(['F', 'M'])]

gender_counts = df_gender.groupby('gendr')['completed'].agg(['sum', 'count'])
print(gender_counts)

stat, pval = proportions_ztest(
    count=gender_counts['sum'].values,
    nobs=gender_counts['count'].values
)
print(f"z={stat:.3f}, p={pval:.4f}")
print(df_gender.groupby('gendr')['completed'].mean())

        sum  count
gendr             
F      8637  15501
M      9512  16169
z=-5.593, p=0.0000
gendr
F    0.557190
M    0.588286
Name: completed, dtype: float64


In [23]:
df_gender = df_cd[df_cd['gendr'].isin(['F', 'M'])]

for gender, group in df_gender.groupby('gendr'):
    counts = group.groupby('group')['completed'].agg(['sum', 'count'])
    stat, pval = proportions_ztest(
        count=counts['sum'].values,
        nobs=counts['count'].values
    )
    rates = group.groupby('group')['completed'].mean()
    print(f"\n{gender}:")
    print(f"  control: {rates['control']:.1%}  test: {rates['test']:.1%}  lift: {rates['test']-rates['control']:.1%}")
    print(f"  z={stat:.3f}, p={pval:.4f}")


F:
  control: 53.3%  test: 57.9%  lift: 4.6%
  z=-5.696, p=0.0000

M:
  control: 55.7%  test: 61.7%  lift: 6.1%
  z=-7.818, p=0.0000
